In [1]:
import os

# Find all files in the dataset
for root, dirs, files in os.walk("/kaggle/input"):
    dirs.sort()
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:  # only show files 3 levels deep to avoid flooding output
        for f in sorted(files)[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} files total)")

input/
  datasets/
    abhishekinnvonix/
      seizure-epilepcy-chb-mit-eeg-dataset-pediatric/
        chb-mit-scalp-eeg-database-1.0.0/
          chb01/
          chb02/
          chb03/
          chb04/
          chb05/
          chb06/
          chb07/
          chb08/
          chb09/
          chb10/
          chb11/
          chb12/
          chb13/
          chb14/
          chb15/
          chb16/
          chb17/
          chb18/
          chb19/
          chb20/
          chb21/
          chb22/
          chb23/
          chb24/


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("github")

In [3]:
# Clone the repo
!git clone https://github.com/tamarajafar/seizure-detection-eeg.git
%cd seizure-detection-eeg

# Install dependencies
!pip install -q mne pyedflib pyyaml tqdm

Cloning into 'seizure-detection-eeg'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 36 (delta 3), reused 33 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 28.26 KiB | 851.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/kaggle/working/seizure-detection-eeg


In [8]:
import re
from pathlib import Path

data_root = Path("/kaggle/input/seizure-epilepcy-chb-mit-eeg-dataset-pediatric")

summary_files = sorted(data_root.rglob("*summary*"))
for sf in summary_files:
    text = sf.read_text()
    n_seizures = len(re.findall(r"Seizure.*?Start", text))
    print(f"{sf.parent.name}: {n_seizures} seizures")

In [11]:
import os

print("=== EVERYTHING IN /kaggle/input/ ===")
for item in sorted(os.listdir("/kaggle/input")):
    print(item)

=== EVERYTHING IN /kaggle/input/ ===
datasets


In [12]:
import os

# Look inside datasets/
for item in sorted(os.listdir("/kaggle/input/datasets")):
    print(item)

abhishekinnvonix


In [13]:
for item in sorted(os.listdir("/kaggle/input/datasets")):
    inner = f"/kaggle/input/datasets/{item}"
    if os.path.isdir(inner):
        print(f"\n{inner}/")
        for sub in sorted(os.listdir(inner))[:5]:
            print(f"  {sub}")


/kaggle/input/datasets/abhishekinnvonix/
  seizure-epilepcy-chb-mit-eeg-dataset-pediatric


In [14]:
import os
from pathlib import Path

data_root = Path("/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric")

# Top level
print("=== TOP LEVEL ===")
for item in sorted(data_root.iterdir()):
    print(item.name)

=== TOP LEVEL ===
chb-mit-scalp-eeg-database-1.0.0


In [15]:
# Look inside the first subject folder
subdirs = sorted([d for d in data_root.iterdir() if d.is_dir()])
if subdirs:
    print(f"=== INSIDE {subdirs[0].name}/ ===")
    for f in sorted(subdirs[0].iterdir())[:15]:
        print(f.name)

=== INSIDE chb-mit-scalp-eeg-database-1.0.0/ ===
ANNOTATORS
RECORDS
RECORDS-WITH-SEIZURES
SHA256SUMS.txt
SUBJECT-INFO
chb01
chb02
chb03
chb04
chb05
chb06
chb07
chb08
chb09
chb10


In [16]:
# Find all .txt files
txt_files = list(data_root.rglob("*.txt"))
print(f"\nFound {len(txt_files)} .txt files")
for f in txt_files[:5]:
    print(f.relative_to(data_root))


Found 25 .txt files
chb-mit-scalp-eeg-database-1.0.0/SHA256SUMS.txt
chb-mit-scalp-eeg-database-1.0.0/chb13/chb13-summary.txt
chb-mit-scalp-eeg-database-1.0.0/chb19/chb19-summary.txt
chb-mit-scalp-eeg-database-1.0.0/chb17/chb17-summary.txt
chb-mit-scalp-eeg-database-1.0.0/chb14/chb14-summary.txt


In [17]:
from pathlib import Path
import re

# Corrected path -- one level deeper than before
data_root = Path("/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0")

# Check all subjects and seizure counts
summary_files = sorted(data_root.rglob("*-summary.txt"))
print(f"Found {len(summary_files)} subjects\n")
for sf in summary_files:
    text = sf.read_text()
    n_seizures = len(re.findall(r"Seizure.*?Start", text))
    print(f"{sf.parent.name}: {n_seizures} seizures")

Found 24 subjects

chb01: 7 seizures
chb02: 3 seizures
chb03: 7 seizures
chb04: 4 seizures
chb05: 5 seizures
chb06: 10 seizures
chb07: 3 seizures
chb08: 5 seizures
chb09: 4 seizures
chb10: 7 seizures
chb11: 3 seizures
chb12: 40 seizures
chb13: 12 seizures
chb14: 8 seizures
chb15: 20 seizures
chb16: 10 seizures
chb17: 3 seizures
chb18: 6 seizures
chb19: 3 seizures
chb20: 8 seizures
chb21: 4 seizures
chb22: 3 seizures
chb23: 7 seizures
chb24: 16 seizures


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/seizure-detection-eeg")

from src.preprocessing.pipeline import run_pipeline

OUT_DIR = Path("/kaggle/working/processed")
run_pipeline(data_root, OUT_DIR)

Preprocessing subjects:   0%|          | 0/24 [00:00<?, ?it/s]


chb01: 42 recordings, 7 seizures


/kaggle/working/seizure-detection-eeg/src/preprocessing/load_edf.py:120: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/kaggle/working/seizure-detection-eeg/src/preprocessing/load_edf.py:120: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/kaggle/working/seizure-detection-eeg/src/preprocessing/load_edf.py:120: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/kaggle/working/seizure-detection-eeg/src/preprocessing/load_edf.py:120: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
